# VASONet

Set the training NPZ and output checkpoint. Select a single element type for the existing VASONet checkpoint architecture.

In [ ]:
from pathlib import Path

DATASET_PATH = Path("data/smoothing_data/vasonet_quad_dataset.npz")
MODEL_PATH = Path("MLP_quad.pt")
SEED = 42
epochs = 200
batch_size = 64

In [ ]:
from vasonet import VASONet

In [ ]:
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

def patch_split(data, generator):
    """Read the saved split or reconstruct the original triangle split."""
    if "split" in data:
        return data["split"]

    group_id = data["group_id"]
    groups = np.unique(group_id)
    groups = groups[torch.randperm(len(groups), generator=generator).numpy()]
    n_train, n_val = int(0.7 * len(groups)), int(0.2 * len(groups))
    split = np.full(len(group_id), 2, dtype=np.int8)
    split[np.isin(group_id, groups[:n_train])] = 0
    split[np.isin(group_id, groups[n_train:n_train + n_val])] = 1
    return split


torch.manual_seed(SEED)
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
generator = torch.Generator().manual_seed(SEED)
with np.load(DATASET_PATH) as data:
    if "element_type" in data:
        raise ValueError("This VASONet trains on one element type at a time. Select a single-type NPZ.")
    features = [torch.as_tensor(data[key], dtype=torch.float64)
                for key in ("coordinates", "valence", "target")]
    split = patch_split(data, generator)

loaders = []
for i in range(3):
    dataset = TensorDataset(*(values[split == i] for values in features))
    loaders.append(DataLoader(dataset, batch_size=batch_size, shuffle=i == 0,
                              generator=generator if i == 0 else None))

In [ ]:
model = VASONet(features[0].shape[1]).to(device)
loss = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=6.5e-4, betas=(0.9, 0.999))
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.916)


def epoch_mse(loader, train=False):
    model.train(train)
    total = 0.0
    with torch.set_grad_enabled(train):
        for x, n, target in loader:
            x, n, target = x.to(device), n.to(device), target.to(device)
            error = loss(model(x, n), target)
            if train:
                optimizer.zero_grad(set_to_none=True)
                error.backward()
                optimizer.step()
            total += error.item() * len(target)
    return total / len(loader.dataset)


for epoch in range(1, epochs + 1):
    train_mse = epoch_mse(loaders[0], train=True)
    val_mse, test_mse = (epoch_mse(loader) for loader in loaders[1:])
    print(f"{epoch:03d}/{epochs} | Train {train_mse:.8f} | Val {val_mse:.8f} | Test {test_mse:.8f}")
    scheduler.step()

torch.save(model.state_dict(), MODEL_PATH)
print(f"Saved {MODEL_PATH}")